In [1]:
# =============================================================================
# Project: Multimodal Sequential Recommendation
# Notebook: 01_Data_Inspection.ipynb
# Author: Manit Chitnukul
# Description:
#   Verifies the feasibility of the dataset by confirming the linkage between
#   Textual Modality (User Reviews) and Visual Modality (Product Images).
# ============================================================================= 

import os 
import gzip
import json
from datetime import datetime

# --- 1. CONFIGURATION & PATH MANAGEMENT ---
# Define paths dynamically to ensure portability across different environments (Dev/Prod/Colab)
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, 'data')

# Target Files: Amazon Luxury Beauty (Raw Data)
# Note: Use the 'Raw' version to endsure to have access to all metadata fields.
REVIEW_FILE = os.path.join(DATA_DIR, 'reviews_Beauty_5.json.gz')
META_FILE = os.path.join(DATA_DIR, 'meta_Beauty.json.gz')

print(f" Data Source Directory: {DATA_DIR}")





 Data Source Directory: /Users/hector/Developments/IS_Multimodal_Recommendation/data


In [ ]:
# --- CORE INSPECTION LOGIC ---
def get_multimodal_sample(limit=5000):
    """
    Attempts to extract a single valid 'Golden Sample' containing both text and image.

    Strategy:
    1. Reverse Indexing (Metadata First): We scan metadata first because images are rarer than reviews.
       building a lookup dictionary of products that HAVE images.
    2. Stream Matching (Reviews Second): Stream the measive review file and check
       if the review's ASIN exists in our image-rich index.
    Args: 
        limit (int): Max number of items to index in memory to provent RAM overflow.
    """
    print("Starting Data Inspection Pipeline...")

    # In-memory index for 0(1) lookup: { asin: metadata_dict }
    meta_index = {}

    # --- Indexing Metadata (find the Images) ---
    print(" Indexing Metadata with Visual Features...")

    if not os.path.exists(META_FILE):
        print(f"Error: Metadata file not found at {META_FILE}")
        return None
    
    try: 
        # Open in Text Mode ('rt') with UTF-8 to handle special characters correctly
        with gzip.open(META_FILE, 'rt', encoding='utf-8') as f:
            for line in f:
                try: 
                    data = json.loads(line.strip())

                    # --- Schema Handling ---
                    # Amazon dataset schema varies. Images can be under 'imageURL' (list) of 'image' (list).
                    # Use .get() to handle both cases safely.

                    raw_imgs = data.get('imageURL', []) or data.get('image', [])

                    # Validation: Item must have an ASIN, a Title, and at least one Image.
                    if 'asin' in data and 'title' in data and raw_imgs:
                        if isinstance(raw_imgs, list) and len(raw_imgs) > 0:
                            # Feature Extraction: We verify we can get the High-Res image URL.
                            # Usually the first element is the main product image.
                            data['evidence_img'] = raw_imgs[0]
                            meta_index[data['asin']] = data 
                    # Optimiztion: Stop indexing early to save time/memory for this inspection task.
                    if len(meta_index) >= limit:
                        break
                except json.JSONDecodeError:
                    continue # Skip malformed JSON lines (common in raw web-scraped data)
    except Exception as e:
        print(f"Critial Error reading metadata: {e}")
        return None

    print(f" Successfully indexed {len(meta_index)} unique items with valid images.") 

    # --- Stream Matching (finding the reviews) ---
    print(" Streaming reviews to find a multimodal match...")

    if not os.path.exists(REVIEW_FILE):
        print(f"Error: Review file not found at {REVIEW_FILE}")
        return None
    try: 
        with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    review = json.loads(line.strip())
                    asin = review.get('asin')

                    # INNER JOIN LOGIC:
                    # Check if the current review corresponds to a product in our image index.
                    if asin in meta_index:
                        product = meta_index[asin]

                        # --- Data fusion ---
                        # Combine Textural Modality (Review) + Visual Modality (Image)
                        evidence = {
                            "asin": asin,
                            "product_title": product['title'],
                            # Truncate text for cleaner display
                            "text_modality": review.get('reviewText', '')[:200] + "...",
                            "visual_modality": product['evidence_img'],
                            "timestamp": datetime.fromtimestamp(review['unixReviewTime']).strftime('%Y-%m-%d') 
                        }

                        # Return immediately upon finding the first valid multimodal instance.
                        return evidence
                except json.JSONDecodeError:
                    continue
    except Exception as e:
        print(f"Critical Error reading reviews: {e}")
        return None
    return None # Return None if no overlapping data is found

# EXECUTION & VERIFICATION ---
if __name__ == "__main__":
    # Run the inspection
    sample_evidence = get_multimodal_sample()

    print("\n" + "="*60)
    print("INSPECTION RESULT: MULTIMODAL FEASIBILITY CHECK")
    print("="*60)

    if sample_evidence:
        print(json.dumps(sample_evidence, indent=4))
        print("\n SUCCESS: Confirmed existence of Text (Review) and Visual (Image) data linkage.")
        print("Status: Ready for Statistical Analysis and Model Development.")
    else: 
        print("\n FAILED: Could not find a matching record. Check dataset paths or schema.")

